## Connection
### Namespace: dimension

|namespace|tableName       |isTemporary|
|---------|----------------|-----------|
|dimension|payment_method  |false      |
|dimension|supplier        |false      |
|dimension|city            |false      |
|dimension|stock_item      |false      |
|dimension|customer        |false      |
|dimension|date            |false      |
|dimension|transaction_type|false      |
|dimension|employee        |false      |



### Namespace: fact

|namespace|tableName    |isTemporary|
|---------|-------------|-----------|
|fact     |purchase     |false      |
|fact     |stock_holding|false      |
|fact     |order        |false      |
|fact     |movement     |false      |
|fact     |sale         |false      |
|fact     |transaction  |false      |



### Namespace: integration

|namespace  |tableName              |isTemporary|
|-----------|-----------------------|-----------|
|integration|employee_staging       |false      |
|integration|transactiontype_staging|false      |
|integration|paymentmethod_staging  |false      |
|integration|supplier_staging       |false      |
|integration|movement_staging       |false      |
|integration|customer_staging       |false      |
|integration|city_staging           |false      |
|integration|stockholding_staging   |false      |
|integration|lineage                |false      |
|integration|order_staging          |false      |
|integration|stockitem_staging      |false      |
|integration|transaction_staging    |false      |
|integration|etl_cutoff             |false      |
|integration|sale_staging           |false      |
|integration|purchase_staging       |false      |



In [ ]:
import os
import sys
from pyspark.sql import functions as sf
from pyspark.sql import window as sw
from pyspark.sql import types as sdt
from pyspark.sql import SparkSession
from datetime import datetime

# 1. Set PYSPARK_SUBMIT_ARGS to match your working batch file launcher
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    '--conf spark.driver.extraClassPath="C:/data/spark/jars/iceberg-spark-runtime-4.0_2.13-1.10.0.jar" '
    "pyspark-shell"
)

# 2. Ensure Python paths align for the worker processes
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

LOCAL_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_warehouse"
STG_WAREHOUSE_PATH = "/data/data_files/iceberg/iceberg_staging_warehouse"
RPT_WAREHOUSE_PATH = "/data/data_files/iceberg/WideWorldImportersDW"

MSSQL_JAR = "C:/data/spark/jars/mssql-jdbc-12.6.5.jre11.jar"

CATALOG_NAME = "local"
STG_CATALOG_NAME = "staging"
WH_CATALOG_NAME = "reporting"

# staging_table_name = "staging.Integration.employee_Staging"
# wh_table_name = "reporting.dimension.Employees"

spark = SparkSession.builder \
    .appName("Iceberg Setup") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", f"file:///{LOCAL_WAREHOUSE_PATH}") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}.warehouse", f"file:///{STG_WAREHOUSE_PATH}") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.warehouse", f"file:///{RPT_WAREHOUSE_PATH}") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .getOrCreate()


spark.catalog.setCurrentCatalog(WH_CATALOG_NAME)

spark
# spark.sql("SHOW CATALOGS").show(truncate=False)
# spark.sql("SHOW NAMESPACES IN reporting").show(truncate=False)
# spark.sql("SHOW DATABASES IN reporting").show(truncate=False)
# spark.sql("SHOW TABLES IN reporting.dimension").show(truncate=False)
for ns in spark.sql("SHOW NAMESPACES IN reporting").collect():
    namespace = ns["namespace"]
    # print(f"\nNamespace: {namespace}")
    spark.sql(f"SHOW TABLES IN reporting.{namespace}").show(truncate=False)


## Remove duplicates

In [ ]:
# List out all the local tables you want to clean up
target_tables = [
    "reporting.Fact.Purchase",
    "reporting.Fact.Stock_Holding",
    "reporting.Fact.Order",
    "reporting.Fact.Movement",
    "reporting.Fact.Sale",
    "reporting.Dimension.payment_method",
    "reporting.Dimension.supplier",
    "reporting.Dimension.city",
    "reporting.Dimension.stock_item",
    "reporting.Dimension.customer",
    "reporting.Dimension.date",
    "reporting.Dimension.transaction_type",
    "reporting.Dimension.employee",
    "reporting.Fact.Transaction"
]


for table_name in target_tables:
    print(f"--- Starting deduplication for: {table_name} ---")
    
    # 1. Read the target table data into memory
    df_raw = spark.table(table_name)
    
    # 2. Extract only the distinct rows
    df_clean = df_raw.dropDuplicates()
    
    # 3. Register the clean dataframe as a local temporary view
    view_name = f"clean_{table_name.replace('.', '_')}"
    df_clean.createOrReplaceTempView(view_name)
    
    # 4. Perform an atomic rewrite using Spark SQL
    # This completely replaces the table contents with the deduplicated dataset
    spark.sql(f"""
        INSERT OVERWRITE {table_name}
        SELECT * FROM {view_name}
    """)
    
    print(f"Successfully cleaned and committed {table_name} to local storage.\n")

## Refined column
- removed spaces in column name
- added underscore

In [ ]:
# Reusable helper to find column modifications
def get_rename_mappings(df):
    """Returns a list of tuples containing (old_name, new_name) for columns with spaces."""
    return [(c, c.replace(" ", "_")) for c in df.columns if " " in c]

# List out all the local tables to clean up
target_tables = [
    "reporting.Fact.Purchase",
    "reporting.Fact.Stock_Holding",
    "reporting.Fact.Order",
    "reporting.Fact.Movement",
    "reporting.Fact.Sale",
    "reporting.Dimension.payment_method",
    "reporting.Dimension.supplier",
    "reporting.Dimension.city",
    "reporting.Dimension.stock_item",
    "reporting.Dimension.customer",
    "reporting.Dimension.date",
    "reporting.Dimension.transaction_type",
    "reporting.Dimension.employee",
    "reporting.Fact.Transaction"
]

for table_name in target_tables:
    try:
        # 1. Load the current metadata schema from the catalog
        df = spark.table(table_name)
        
        # 2. Identify which columns actually need updating
        renames = get_rename_mappings(df)
        
        if not renames:
            print(f"✅ {table_name}: No spaces found in column names. Skipping.")
            continue
            
        print(f"🔄 {table_name}: Found {len(renames)} columns to sanitize...")
        
        # 3. Alter the table schema permanently in the catalog
        for old_name, new_name in renames:
            # Escape columns with spaces using backticks for the SQL parser
            alter_query = f"ALTER TABLE {table_name} RENAME COLUMN `{old_name}` TO `{new_name}`"
            spark.sql(alter_query)
            print(f"   ↳ Renamed: '{old_name}' -> '{new_name}'")
            
        print(f"🎉 {table_name}: All columns permanently sanitized.\n")
        
    except Exception as e:
        print(f"❌ Error processing table {table_name}: {str(e)}\n")

## Listout All columns

In [ ]:
# Clear all cached relational plans and tables in this session
spark.catalog.clearCache()

# Force spark to re-read metadata schemas directly from storage paths
for table in target_tables:
    try:
        spark.catalog.refreshTable(table)
    except:
        pass

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# 1. Define the schema for our master metadata list
metadata_schema = StructType([
    StructField("TableName", StringType(), False),
    StructField("ColumnName", StringType(), False),
    StructField("ColumnOrder", IntegerType(), False),
    StructField("DataType", StringType(), False)
])

# Initialize an empty list to accumulate rows
metadata_rows = []

target_tables = [
    "reporting.Fact.Purchase",
    "reporting.Fact.Stock_Holding",
    "reporting.Fact.Order",
    "reporting.Fact.Movement",
    "reporting.Fact.Sale",
    "reporting.Dimension.payment_method",
    "reporting.Dimension.supplier",
    "reporting.Dimension.city",
    "reporting.Dimension.stock_item",
    "reporting.Dimension.customer",
    "reporting.Dimension.date",
    "reporting.Dimension.transaction_type",
    "reporting.Dimension.employee",
    "reporting.Fact.Transaction"
]

# 2. Loop through each table and extract structural metadata
for table_name in target_tables:
    try:
        # Load table metadata lazily (no data is pulled into memory)
        df = spark.table(table_name)
        
        # Iterate over columns with index tracking for ColumnOrder
        for index, field in enumerate(df.schema, start=1):
            metadata_rows.append((
                table_name,
                field.name,
                index,
                field.dataType.simpleString() # Returns human-readable types like int, string, timestamp
            ))
            
    except Exception as e:
        print(f"⚠️ Could not read schema for {table_name}: {e}")

# 3. Convert the accumulated records back into a clean Spark DataFrame
df_schema_manifest = spark.sql("SELECT NULL")

df_schema_manifest = df_schema_manifest.withColumn("TableName", sf.lit(None).cast(StringType())) \
    .withColumn("ColumnName", sf.lit(None).cast(StringType())) \
    .withColumn("ColumnOrder", sf.lit(None).cast(IntegerType())) \
    .withColumn("DataType", sf.lit(None).cast(StringType())) \
    .drop("NULL") # Clean up that initial junk column

# --- The Fix: Insert metadata_rows into the Dataframe ---
if metadata_rows:
    # 3.1. Map your Python list into an SQL string
    sql_values = ", ".join(
        f"('{table}', '{col}', {order}, '{dtype}')" 
        for table, col, order, dtype in metadata_rows
    )
    
    # 3.2. Convert the text values directly into a temporary DataFrame using Spark SQL
    df_data_to_insert = spark.sql(f"""
        SELECT 
            col1 AS TableName, 
            col2 AS ColumnName, 
            col3 AS ColumnOrder, 
            col4 AS DataType 
        FROM VALUES {sql_values}
    """)
    
# 4. Insert/Append into your manifest frame safely aligning columns by name
df_schema_manifest = df_schema_manifest.unionByName(df_data_to_insert)
df_schema_manifest = df_schema_manifest.where(sf.col("TableName").isNotNull())
# --- 4. Verify the results ---
df_schema_manifest.show(100, truncate=False)

In [ ]:
# Save as a standard CSV with a pipe (|) delimiter so it reads like a clean text table
df_schema_manifest.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", "true") \
    .option("delimiter", "|") \
    .csv("local_schema_manifest_output")

print("Exported successfully! Check the folder 'local_schema_manifest_output' for the text/csv file.")

## Loaded table into data frame

In [ ]:
df_purchase = spark.table("reporting.Fact.Purchase").alias("Purchase")
df_stock_holding = spark.table("reporting.Fact.Stock_Holding").alias("Stock_Holding")
df_order = spark.table("reporting.Fact.Order").alias("Order")
df_movement = spark.table("reporting.Fact.Movement").alias("Movement")
df_sale = spark.table("reporting.Fact.Sale").alias("Sale")
df_payment_method = spark.table("reporting.Dimension.payment_method").alias("payment_method")
df_supplier = spark.table("reporting.Dimension.supplier").alias("supplier")
df_city = spark.table("reporting.Dimension.city").alias("city")
df_stock_item = spark.table("reporting.Dimension.stock_item").alias("stock_item")
df_customer = spark.table("reporting.Dimension.customer").alias("customer")
df_date = spark.table("reporting.Dimension.date").alias("date")
df_transaction_type = spark.table("reporting.Dimension.transaction_type").alias("transaction_type")
df_employee = spark.table("reporting.Dimension.employee").alias("employee")
df_transaction = spark.table("reporting.Fact.Transaction").alias("Transaction")

## Update Dates

### Update Date Dimension

In [ ]:
spark.sql("""
UPDATE reporting.Dimension.date
SET 
    Date = add_months(Date, 120),
    Day_Number = day(add_months(Date, 120)),
    Day = date_format(add_months(Date, 120), 'EEEE'),
    Month = date_format(add_months(Date, 120), 'MMMM'),
    Short_Month = date_format(add_months(Date, 120), 'MMM'),
    Calendar_Month_Number = month(add_months(Date, 120)),
    Calendar_Month_Label = date_format(add_months(Date, 120), 'yyyy-MMM'),
    Calendar_Year = year(add_months(Date, 120)),
    Calendar_Year_Label = concat('CY', year(add_months(Date, 120))),
    ISO_Week_Number = weekofyear(add_months(Date, 120))
""")

### Testing

In [ ]:
print(df_sale.count())
print(df_customer.count())
df_city.show(100, truncate=False)

## Check Date

In [ ]:
df_date.show(100, truncate=False)

In [ ]:
import pyspark.sql.functions as sf

df_distinct_months = df_sale.select(
    sf.year("Invoice_Date_Key").alias("Year"),
    sf.month("Invoice_Date_Key").alias("Month")
).distinct().orderBy("Year", "Month")

df_distinct_months.show(560, truncate=False)

## Data Analysis

### Customer wise **Sale**

In [ ]:
import pyspark.sql.functions as sf

# 1. Perform Joins
df_joined = df_customer.join(df_sale, df_customer["Customer_Key"] == df_sale["Customer_Key"], "inner") \
    .join(df_date, df_sale["Invoice_Date_Key"] == df_date["Date"], "inner") \
    .join(df_stock_item, df_sale["Stock_Item_Key"] == df_stock_item["Stock_Item_Key"], "inner")

# 2. Filter early using the available date columns
df_filtered = df_joined.filter(
    (sf.col("Calendar_Year") == 2016) & (sf.col("Calendar_Month_Number") == 4)
)

# 3. Select, GroupBy, and Aggregate (Fixing the ambiguity here)
df_customer_sales = df_filtered.select(
    sf.year(df_sale["Invoice_Date_Key"]).alias("SaleYear"),
    sf.month(df_sale["Invoice_Date_Key"]).alias("SaleMonth"),
    sf.col("Customer").alias("CustomerName"), 
    sf.col("Stock_Item").alias("ItemName"),
    # FIXED: Explicitly use df_sale["Unit_Price"] to resolve the ambiguity
    (df_sale["Unit_Price"] * sf.col("Quantity")).alias("SaleAmount")
) \
.rollup(
    "SaleYear",
    "SaleMonth",
    "CustomerName",
    "ItemName"
) \
.agg(
    sf.sum("SaleAmount").alias("TotalSales")
) \
.orderBy(
    sf.col("SaleYear").asc_nulls_last(),
    sf.col("SaleMonth").asc_nulls_last(),
    sf.col("CustomerName").asc_nulls_last(),
    sf.col("ItemName").asc_nulls_last()
)

# 4. Display results
df_customer_sales.show(100, truncate=False)

### Inward Outward Report

In [ ]:
import pyspark.sql.functions as sf

# 1. Perform Joins
df_Sale_Purchase = df_stock_item.join(df_sale, df_stock_item["Stock_Item_Key"] == df_sale["Stock_Item_Key"], "left") \
    .join(df_purchase, df_stock_item["Stock_Item_Key"] == df_purchase["Stock_Item_Key"], "left") \
    .join(df_date, (df_sale["Invoice_Date_Key"] == df_date["Date"]) & (df_purchase["Date_key"] == df_date["Date"]), "left") \
    

# 2. Filter early using the available date columns
df_Sale_Purchase = df_Sale_Purchase.filter(
    (sf.col("Calendar_Year") == 2016) & (sf.col("Calendar_Month_Number") == 4)
)


df_Sale_Purchase = df_Sale_Purchase.select (
    df_stock_item["Stock_Item"].alias("ItemName"),
    df_stock_item["Color"].alias("ItemColor"),
    df_stock_item["Brand"].alias("ItemBrand"),
    df_stock_item["Size"].alias("ItemSize"),
    df_sale["Invoice_Date_Key"].alias("SaleDate"),
    df_purchase["Date_key"].alias("PurchaseDate"),
    df_sale["Unit_Price"].alias("SaleUnitPrice"),
    df_stock_item["Unit_Price"].alias("PurchaseUnitPrice"),
    df_sale["Quantity"].alias("SaleQuantity"),
    df_purchase["Ordered_Quantity"].alias("PurchaseQuantity"),
    (df_sale["Unit_Price"] * df_sale["Quantity"]).alias("SaleAmount"),
    (df_stock_item["Unit_Price"] * df_purchase["Ordered_Quantity"]).alias("PurchaseAmount")
)

df_Sale_Purchase.show(100, truncate=False)
 


In [ ]:
spark.stop()